# Stocks — Seven-Trading-Session Move After Quarterly Results

Measure the next seven trading sessions after every declaration in `nifty50_quaterly_result_dates.csv`, using the local daily stock candles. No downloads are required.

**Default definition:** the next **seven observed market sessions strictly after the result date**. The reference price is the last available close on or before that date. For weekend/holiday announcements this is the preceding session's close. The primary **weekly move** (seven trading sessions, per the selected horizon) includes the opening gap; **post-entry move** instead starts at the first post-results session's open.

Announcement times are absent: this convention assumes results are available after the result-date close, and cannot isolate reactions to announcements before or during that session. Same-date OHLC is excluded. This is a descriptive event study, not an estimate of a causal earnings effect or an option-return backtest.

Change the parameters below to inspect a stock, choose another week definition, or restrict result dates. All 50 CSV symbols remain in the coverage report, including symbols without prices.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display


def find_project_root():
    for directory in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (directory / "data/raw/option_selling/stocks").is_dir():
            return directory
    raise FileNotFoundError("Run from the project root or a notebook directory inside it")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data/raw/option_selling/stocks"
RESULTS_CSV = RAW_DIR / "nifty50_quaterly_result_dates.csv"
EXPORT_DIR = PROJECT_ROOT / "data/processed/option_selling/stocks/quarterly_results_weekly_move"

SYMBOL = "RELIANCE"                 # Detail table/charts; all stocks are analyzed.
WINDOW_MODE = "trading_sessions"       # trading_sessions, seven_calendar_days, following_monday_friday
TRADING_SESSIONS = 7                  # Next seven trading sessions after declaration.
START_RESULT_DATE = None            # Inclusive, e.g. "2022-01-01"
END_RESULT_DATE = None              # Inclusive, e.g. "2025-12-31"
LARGE_DAILY_MOVE_PCT = 25.0          # Flag large close-to-close OR overnight moves.
EXCLUDE_LARGE_DAILY_MOVES = False    # Retain flagged events by default; inspect them below.
BANDS_PCT = [1.0, 2.0, 3.0, 4.0, 5.0, 7.5, 10.0]
EXPORT_CSV = True

if WINDOW_MODE not in {"trading_sessions", "seven_calendar_days", "following_monday_friday"}:
    raise ValueError("Unknown WINDOW_MODE")
if not isinstance(TRADING_SESSIONS, int) or isinstance(TRADING_SESSIONS, bool) or TRADING_SESSIONS < 1:
    raise ValueError("TRADING_SESSIONS must be a positive integer")
if LARGE_DAILY_MOVE_PCT <= 0 or any(distance <= 0 for distance in BANDS_PCT):
    raise ValueError("Move thresholds must be positive")
if START_RESULT_DATE and END_RESULT_DATE and pd.Timestamp(START_RESULT_DATE) > pd.Timestamp(END_RESULT_DATE):
    raise ValueError("START_RESULT_DATE must precede END_RESULT_DATE")

HORIZON_LABEL = f"Next {TRADING_SESSIONS} trading sessions" if WINDOW_MODE == "trading_sessions" else WINDOW_MODE.replace("_", " ")

pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.options.display.float_format = "{:,.2f}".format


def scroll_table(frame, height=430):
    display(HTML(
        f'<div style="max-height:{height}px;overflow:auto;border:1px solid #ddd">'
        + frame.to_html(index=False, float_format=lambda value: f"{value:,.2f}", na_rep="—")
        + '</div>'
    ))

## Load result dates and daily prices

File symbols are matched case-insensitively; no company/renaming aliases are invented. If several files match a symbol, the file with the latest end date (then earliest start date) is selected and its actual coverage is reported. CSV notes are preserved; these dates have not been independently verified.

Exact duplicate candles are removed; conflicting dates, invalid OHLC values and duplicate result events raise errors. The **union of session dates in the selected price files** serves as the observed market calendar. Missing sessions in one stock are flagged rather than replaced with later days. Dates missing from every file cannot be detected without an external exchange calendar. Weekend sessions present in the source are retained.

In [2]:
results = pd.read_csv(RESULTS_CSV, keep_default_na=False)
required = {"company", "nse_symbol", "fiscal_quarter", "quarter_ended", "result_date", "note"}
if not required.issubset(results.columns):
    raise ValueError(f"Missing result columns: {required - set(results.columns)}")
results["nse_symbol"] = results["nse_symbol"].str.strip().str.upper()
for column in ["result_date", "quarter_ended"]:
    results[column] = pd.to_datetime(results[column], format="%Y-%m-%d", errors="raise")
    if results[column].isna().any():
        raise ValueError(f"Missing {column}")
if results["nse_symbol"].eq("").any() or results.duplicated(["nse_symbol", "result_date"]).any():
    raise ValueError("Blank symbols or duplicate stock/result-date events")
if START_RESULT_DATE:
    results = results.loc[results["result_date"] >= pd.Timestamp(START_RESULT_DATE)]
if END_RESULT_DATE:
    results = results.loc[results["result_date"] <= pd.Timestamp(END_RESULT_DATE)]
results = results.sort_values(["nse_symbol", "result_date"]).reset_index(drop=True)
if results.empty:
    raise ValueError("No result dates match the selected filters")
SYMBOL = SYMBOL.strip().upper()
if SYMBOL not in set(results["nse_symbol"]):
    raise ValueError(f"Choose SYMBOL from: {', '.join(sorted(results['nse_symbol'].unique()))}")

candidates = {}
for path in sorted(RAW_DIR.glob("*.json")):
    parts = path.stem.rsplit("_", 2)
    if len(parts) == 3:
        symbol, start, end = parts
        candidates.setdefault(symbol.upper(), []).append((pd.Timestamp(end), pd.Timestamp(start), path))
source_files = {
    symbol: sorted(options, key=lambda item: (-item[0].value, item[1].value, item[2].name))[0][2]
    for symbol, options in candidates.items()
}


def load_candles(path):
    candles = json.loads(path.read_text())["data"]["candles"]
    if not candles or any(len(row) != 7 for row in candles):
        raise ValueError(f"{path.name}: expected nonempty seven-field candles")
    frame = pd.DataFrame(candles, columns=["timestamp", "open", "high", "low", "close", "volume", "oi"])
    frame["timestamp"] = (pd.to_datetime(frame["timestamp"], utc=True, errors="raise")
                          .dt.tz_convert("Asia/Kolkata").dt.tz_localize(None).dt.normalize())
    count = len(frame)
    frame = frame.drop_duplicates()
    if frame["timestamp"].isna().any() or frame["timestamp"].duplicated().any():
        raise ValueError(f"{path.name}: missing/conflicting session dates")
    price = frame.set_index("timestamp").sort_index()[["open", "high", "low", "close"]].apply(pd.to_numeric, errors="raise")
    valid = (np.isfinite(price).all(axis=1) & price.gt(0).all(axis=1)
             & price["high"].ge(price.max(axis=1)) & price["low"].le(price.min(axis=1)))
    if not valid.all():
        raise ValueError(f"{path.name}: {int((~valid).sum())} invalid OHLC rows")
    return price, count - len(frame)


prices, coverage_rows = {}, []
for symbol, group in results.groupby("nse_symbol"):
    path = source_files.get(symbol)
    row = {"nse_symbol": symbol, "result_events": len(group), "source_file": path.name if path else "",
           "candidate_files": len(candidates.get(symbol, [])), "price_status": "missing price file"}
    if path:
        price, removed = load_candles(path)
        prices[symbol] = price
        row.update(price_status="loaded", first_session=price.index.min(), last_session=price.index.max(),
                   sessions=len(price), exact_duplicates_removed=removed)
    coverage_rows.append(row)
coverage = pd.DataFrame(coverage_rows)
if not prices:
    raise ValueError("No matching stock price files")
market_sessions = pd.DatetimeIndex(sorted(set().union(*(set(price.index) for price in prices.values()))))
print(f"{len(results):,} result events; {results['nse_symbol'].nunique()} stocks; {len(prices)} price files loaded")
scroll_table(coverage)

1,326 result events; 50 stocks; 49 price files loaded


nse_symbol,result_events,source_file,candidate_files,price_status,first_session,last_session,sessions,exact_duplicates_removed
ADANIENT,27,adanient_2020-01-01_2026-09-23.json,1,loaded,2020-01-01,2026-09-22,"1,671.00",0.00
ADANIPORTS,27,adaniports_2020-01-01_2026-09-23.json,1,loaded,2020-01-01,2026-09-22,"1,671.00",0.00
APOLLOHOSP,27,apollohosp_2020-01-01_2026-09-24.json,1,loaded,2020-01-01,2026-09-23,"1,672.00",0.00
ASIANPAINT,27,asianpaint_2020-01-01_2026-09-23.json,1,loaded,2020-01-01,2026-09-22,"1,671.00",0.00
AXISBANK,27,axisbank_2020-01-01_2026-09-23.json,1,loaded,2020-01-01,2026-09-22,"1,671.00",0.00
BAJAJ-AUTO,27,bajaj-auto_2020-01-01_2026-09-23.json,1,loaded,2020-01-01,2026-09-22,"1,671.00",0.00
BAJAJFINSV,27,bajajfinsv_2020-01-01_2026-09-23.json,1,loaded,2020-01-01,2026-09-22,"1,671.00",0.00
BAJFINANCE,27,bajfinance_2020-01-01_2026-09-23.json,1,loaded,2020-01-01,2026-09-22,"1,671.00",0.00
BEL,27,bel_2020-01-01_2026-09-23.json,1,loaded,2020-01-01,2026-09-22,"1,671.00",0.00
BHARTIARTL,27,bhartiartl_2020-01-01_2026-09-23.json,1,loaded,2020-01-01,2026-09-22,"1,671.00",0.00


## Measure each event window

- `trading_sessions`: next `TRADING_SESSIONS` observed market sessions strictly after declaration (default: **7**).
- `seven_calendar_days`: observed sessions in `(result_date, result_date + 7 days]`.
- `following_monday_friday`: Monday through Friday of the next calendar week; a Monday declaration starts the following Monday.

`weekly_move_pct = (end_close / base_close − 1) × 100`. The baseline is the last market close on or before declaration. `post_entry_move_pct` uses the first window session's open, and `opening_gap_pct` measures that open versus the baseline (for the following-week mode, this can span intervening sessions).

High/low moves are signed distances from the baseline, so the high can remain negative or the low positive. `range_pct` is `(weekly_high − weekly_low) / base_close × 100`. The final close is the window end, not an options expiry.

Incomplete windows and missing baselines remain in the audit with no return. Large price jumps may reflect either real market movement or unadjusted corporate actions; they are flagged, never automatically adjusted. Source prices are used as supplied and adjustment status is unknown.

In [3]:
def event_sessions(result_date, calendar, mode, trading_sessions=TRADING_SESSIONS):
    if mode == "trading_sessions":
        sessions = calendar[calendar > result_date][:trading_sessions]
        return sessions, len(sessions) == trading_sessions
    if mode == "seven_calendar_days":
        start, end = result_date + pd.Timedelta(days=1), result_date + pd.Timedelta(days=7)
    elif mode == "following_monday_friday":
        start = result_date + pd.Timedelta(days=7 - result_date.dayofweek)
        end = start + pd.Timedelta(days=4)
    else:
        raise ValueError(f"Unknown window mode: {mode}")
    return calendar[(calendar >= start) & (calendar <= end)], end <= calendar.max()


def measure_event(event, price, calendar, mode):
    row = dict(event)
    row.update(window_mode=mode, requested_trading_sessions=TRADING_SESSIONS if mode == "trading_sessions" else np.nan, included=False, status="", large_daily_move=False,
               base_date=pd.NaT, window_start=pd.NaT, window_end=pd.NaT,
               expected_sessions=0, observed_sessions=0, missing_sessions="",
               base_close=np.nan, entry_open=np.nan, end_close=np.nan,
               weekly_high=np.nan, weekly_low=np.nan, high_date=pd.NaT, low_date=pd.NaT,
               weekly_move_points=np.nan, weekly_move_pct=np.nan, post_entry_move_pct=np.nan,
               opening_gap_pct=np.nan, max_up_pct=np.nan, max_down_pct=np.nan,
               range_pct=np.nan, largest_daily_move_pct=np.nan)
    expected, full_window = event_sessions(event["result_date"], calendar, mode)
    row["expected_sessions"] = len(expected)
    if len(expected):
        row.update(window_start=expected[0], window_end=expected[-1])
    if price is None:
        row["status"] = "missing price file"
        return row
    prior = calendar[calendar <= event["result_date"]]
    if not len(prior) or prior[-1] not in price.index:
        row["status"] = "missing baseline session"
        return row
    base_date = prior[-1]
    row.update(base_date=base_date, base_close=float(price.at[base_date, "close"]))
    missing = expected.difference(price.index)
    row.update(observed_sessions=len(expected.intersection(price.index)),
               missing_sessions=", ".join(missing.strftime("%Y-%m-%d")))
    issues = []
    if not full_window:
        issues.append("incomplete market window at data end")
    if not len(expected):
        issues.append("no post-results sessions")
    if len(missing):
        issues.append("missing stock sessions")
    if issues:
        row["status"] = "; ".join(issues)
        return row
    window = price.loc[expected]
    base = row["base_close"]
    entry, close = float(window.iloc[0]["open"]), float(window.iloc[-1]["close"])
    high, low = float(window["high"].max()), float(window["low"].min())
    # Check every intervening session, including any days before a following-week window.
    path = price.loc[base_date:expected[-1]]
    previous_close = path["close"].shift()
    jumps = pd.concat([(path["close"] / previous_close - 1).abs(),
                       (path["open"] / previous_close - 1).abs()]) * 100
    largest_jump = jumps.max()
    flagged = bool(largest_jump > LARGE_DAILY_MOVE_PCT)
    row.update(entry_open=entry, end_close=close, weekly_high=high, weekly_low=low,
               high_date=window["high"].idxmax(), low_date=window["low"].idxmin(),
               weekly_move_points=close - base, weekly_move_pct=(close / base - 1) * 100,
               post_entry_move_pct=(close / entry - 1) * 100,
               opening_gap_pct=(entry / base - 1) * 100,
               max_up_pct=(high / base - 1) * 100, max_down_pct=(low / base - 1) * 100,
               range_pct=(high - low) / base * 100,
               largest_daily_move_pct=largest_jump, large_daily_move=flagged,
               included=not (flagged and EXCLUDE_LARGE_DAILY_MOVES),
               status="excluded: large daily move" if flagged and EXCLUDE_LARGE_DAILY_MOVES else "included")
    return row


audit = pd.DataFrame([
    measure_event(event, prices.get(event["nse_symbol"]), market_sessions, WINDOW_MODE)
    for event in results.to_dict("records")
])
weekly = audit.loc[audit["included"]].copy()
weekly["abs_move_pct"] = weekly["weekly_move_pct"].abs()
print(f"Window: {WINDOW_MODE} ({TRADING_SESSIONS} sessions when trading_sessions); included: {len(weekly):,}; excluded: {len(audit) - len(weekly):,}")
display(audit.groupby("status").size().rename("events").to_frame())
scroll_table(audit.loc[~audit["included"], [
    "nse_symbol", "fiscal_quarter", "result_date", "status", "missing_sessions", "note"
]], height=280)
if weekly.empty:
    print("No complete events. Audit/coverage and CSV exports remain available; charts will be skipped.")

Window: trading_sessions (7 sessions when trading_sessions); included: 1,299; excluded: 27


,events
status,
included,1299
missing price file,27


nse_symbol,fiscal_quarter,result_date,status,missing_sessions,note
TATACONSUM,Q3 FY20,2020-02-04,missing price file,,
TATACONSUM,Q4 FY20,2020-05-14,missing price file,,
TATACONSUM,Q1 FY21,2020-08-04,missing price file,,
TATACONSUM,Q2 FY21,2020-11-06,missing price file,,
TATACONSUM,Q3 FY21,2021-02-02,missing price file,,
TATACONSUM,Q4 FY21,2021-05-06,missing price file,,
TATACONSUM,Q1 FY22,2021-08-03,missing price file,,
TATACONSUM,Q2 FY22,2021-10-22,missing price file,,
TATACONSUM,Q3 FY22,2022-02-02,missing price file,,
TATACONSUM,Q4 FY22,2022-05-04,missing price file,,


## Stock-by-stock summary

The signed average can cancel gains and losses; mean absolute move measures size. P90 is the historical 90th percentile, not a forecast. Each stock has its own sample count. Stocks with no complete events retain a row with empty statistics.

In [4]:
summary = coverage[["nse_symbol", "result_events", "price_status"]].set_index("nse_symbol").copy()
stats = weekly.groupby("nse_symbol").agg(
    included_events=("weekly_move_pct", "size"),
    first_result=("result_date", "min"), last_result=("result_date", "max"),
    mean_move_pct=("weekly_move_pct", "mean"), median_move_pct=("weekly_move_pct", "median"),
    mean_abs_move_pct=("abs_move_pct", "mean"),
    p90_abs_move_pct=("abs_move_pct", lambda values: values.quantile(0.9)),
    best_move_pct=("weekly_move_pct", "max"), worst_move_pct=("weekly_move_pct", "min"),
    up_events_pct=("weekly_move_pct", lambda values: values.gt(0).mean() * 100),
    mean_max_up_pct=("max_up_pct", "mean"), mean_max_down_pct=("max_down_pct", "mean"),
    mean_post_entry_move_pct=("post_entry_move_pct", "mean"),
    flagged_events=("large_daily_move", "sum"),
)
summary = summary.join(stats)
summary["included_events"] = summary["included_events"].fillna(0).astype(int)
summary["excluded_events"] = summary["result_events"] - summary["included_events"]
summary = summary.sort_values("mean_abs_move_pct", ascending=False)
scroll_table(summary.reset_index(), height=600)

if not weekly.empty:
    chart_data = summary.loc[summary["included_events"] > 0].reset_index().sort_values("mean_abs_move_pct")
    fig = px.bar(chart_data, x="mean_abs_move_pct", y="nse_symbol", orientation="h",
                 hover_data=["included_events", "mean_move_pct", "p90_abs_move_pct", "flagged_events"],
                 labels={"mean_abs_move_pct": "Mean absolute weekly move (%)", "nse_symbol": "Stock"},
                 title=f"{HORIZON_LABEL} after results — mean absolute move")
    fig.update_layout(height=max(500, 22 * len(chart_data)), margin=dict(l=90))
    fig.show()

nse_symbol,result_events,price_status,included_events,first_result,last_result,mean_move_pct,median_move_pct,mean_abs_move_pct,p90_abs_move_pct,best_move_pct,worst_move_pct,up_events_pct,mean_max_up_pct,mean_max_down_pct,mean_post_entry_move_pct,flagged_events,excluded_events
ETERNAL,21,loaded,21,2021-08-10,2026-07-22,4.68,3.64,8.47,13.66,30.94,-13.66,71.43,10.94,-3.63,3.35,0.00,0
TRENT,27,loaded,27,2020-02-06,2026-08-06,3.69,3.03,7.66,11.68,30.17,-9.89,59.26,8.34,-3.50,2.92,0.00,0
SHRIRAMFIN,27,loaded,27,2020-02-03,2026-07-24,4.05,2.25,6.64,15.94,27.77,-8.24,59.26,8.73,-3.19,2.68,0.00,0
AXISBANK,27,loaded,27,2020-01-22,2026-07-18,-0.82,-0.33,6.42,13.62,13.84,-16.12,48.15,4.03,-5.01,-0.56,0.00,0
TMPV,27,loaded,27,2020-01-30,2026-08-13,1.50,1.47,6.25,11.39,23.73,-10.07,59.26,5.55,-4.43,1.59,0.00,0
BSE,27,loaded,27,2020-02-11,2026-08-04,0.25,-2.13,6.06,10.64,14.46,-10.33,44.44,5.94,-4.74,-0.86,0.00,0
BAJFINANCE,27,loaded,27,2020-01-29,2026-07-30,-0.01,0.65,6.05,12.28,14.23,-17.13,51.85,4.65,-4.85,0.30,0.00,0
APOLLOHOSP,27,loaded,27,2020-02-13,2026-08-12,4.09,4.12,6.00,13.19,17.41,-8.10,81.48,8.60,-2.19,3.57,0.00,0
BEL,27,loaded,27,2020-01-30,2026-07-27,2.60,1.02,5.61,12.97,22.00,-13.10,66.67,6.05,-4.44,1.94,0.00,0
EICHERMOT,27,loaded,27,2020-02-06,2026-07-29,2.39,3.71,5.59,10.10,11.78,-10.91,74.07,5.98,-2.31,1.58,0.00,0


## Selected stock — every quarterly result

Change `SYMBOL` in the parameter cell and rerun. The table includes excluded events and original notes; plots use only included events. High/low traces show movement during the week alongside the final return.

In [5]:
selected_audit = audit.loc[audit["nse_symbol"].eq(SYMBOL)].sort_values("result_date", ascending=False)
selected = weekly.loc[weekly["nse_symbol"].eq(SYMBOL)].sort_values("result_date")
print(f"{SYMBOL}: {len(selected)} included / {len(selected_audit)} result dates")
scroll_table(selected_audit[[
    "fiscal_quarter", "result_date", "base_date", "window_start", "window_end", "observed_sessions",
    "base_close", "entry_open", "end_close", "weekly_move_points", "weekly_move_pct",
    "opening_gap_pct", "post_entry_move_pct", "max_up_pct", "max_down_pct", "range_pct",
    "large_daily_move", "status", "note",
]], height=540)

if not selected.empty:
    fig = go.Figure()
    for column, label, color in [
        ("max_up_pct", "Weekly high vs baseline", "#1baf7a"),
        ("max_down_pct", "Weekly low vs baseline", "#d75555"),
        ("weekly_move_pct", "Week-end close vs baseline", "#2a78d6"),
        ("post_entry_move_pct", "Week-end close vs entry open", "#eb6834"),
    ]:
        fig.add_trace(go.Scatter(
            x=selected["result_date"], y=selected[column], name=label, mode="lines+markers",
            line=dict(color=color), customdata=selected[["fiscal_quarter"]],
            hovertemplate="%{x|%Y-%m-%d} · %{customdata[0]}<br>%{y:+.2f}%<extra>%{fullData.name}</extra>",
        ))
    fig.add_hline(y=0, line_color="#999", line_width=1)
    fig.update_layout(title=f"{SYMBOL} — {HORIZON_LABEL.lower()} after each result", yaxis_title="Move (%)",
                      xaxis_title="Result declaration date", height=460, hovermode="x unified")
    fig.show()
else:
    print("No complete windows for this stock; see the audit statuses above.")

RELIANCE: 27 included / 27 result dates


fiscal_quarter,result_date,base_date,window_start,window_end,observed_sessions,base_close,entry_open,end_close,weekly_move_points,weekly_move_pct,opening_gap_pct,post_entry_move_pct,max_up_pct,max_down_pct,range_pct,large_daily_move,status,note
Q1 FY27,2026-07-17,2026-07-17,2026-07-20,2026-07-28,7,"1,327.20","1,317.20","1,267.70",-59.50,-4.48,-0.75,-3.76,1.41,-5.83,7.24,False,included,
Q4 FY26,2026-04-24,2026-04-24,2026-04-27,2026-05-06,7,"1,327.80","1,313.00","1,437.90",110.10,8.29,-1.11,9.51,10.97,-1.27,12.23,False,included,
Q3 FY26,2026-01-16,2026-01-16,2026-01-19,2026-01-28,7,"1,457.90","1,450.60","1,396.70",-61.20,-4.20,-0.50,-3.72,-0.50,-6.17,5.67,False,included,
Q2 FY26,2025-10-17,2025-10-17,2025-10-20,2025-10-29,7,"1,416.80","1,440.00","1,504.20",87.40,6.17,1.64,4.46,6.46,1.16,5.29,False,included,
Q1 FY26,2025-07-18,2025-07-18,2025-07-21,2025-07-29,7,"1,476.00","1,465.00","1,417.10",-58.90,-3.99,-0.75,-3.27,0.00,-6.30,6.30,False,included,
Q4 FY25,2025-04-25,2025-04-25,2025-04-28,2025-05-07,7,"1,300.40","1,340.00","1,406.00",105.60,8.12,3.05,4.93,10.70,1.52,9.17,False,included,
Q3 FY25,2025-01-16,2025-01-16,2025-01-17,2025-01-27,7,"1,266.45","1,322.25","1,229.35",-37.10,-2.93,4.41,-7.03,4.70,-3.67,8.37,False,included,
Q2 FY25,2024-10-14,2024-10-14,2024-10-15,2024-10-23,7,"1,372.50","1,358.50","1,338.50",-34.00,-2.48,-1.02,-1.47,0.26,-2.73,2.99,False,included,
Q1 FY25,2024-07-19,2024-07-19,2024-07-22,2024-07-30,7,"1,555.15","1,537.25","1,513.15",-42.00,-2.70,-1.15,-1.57,-1.13,-5.93,4.79,False,included,
Q4 FY24,2024-04-22,2024-04-22,2024-04-23,2024-05-02,7,"1,479.85","1,479.00","1,466.55",-13.30,-0.90,-0.06,-0.84,0.92,-2.59,3.51,False,included,


## Weekly price bands and data-quality review

`close_inside_pct` counts week-end returns within ± the band around the baseline. `range_inside_pct` requires every daily high/low in the window to stay inside. Boundary values count as inside. Frequencies describe observed stock prices; they do not include option premiums or transaction costs.

The review table keeps all events with a source note or a large daily jump. Notes can describe renamed companies, inferred dates or other provenance, so a note alone does not exclude an event. Review flagged jumps before interpreting extreme returns; this threshold does not detect all corporate actions.

In [6]:
band_rows = []
for symbol, group in weekly.groupby("nse_symbol"):
    for distance in BANDS_PCT:
        band_rows.append({
            "nse_symbol": symbol, "band_pct": distance, "events": len(group),
            "close_inside_pct": group["weekly_move_pct"].abs().le(distance).mean() * 100,
            "range_inside_pct": (group["max_up_pct"].le(distance)
                                 & group["max_down_pct"].ge(-distance)).mean() * 100,
        })
bands = pd.DataFrame(band_rows, columns=["nse_symbol", "band_pct", "events", "close_inside_pct", "range_inside_pct"])
display(bands.loc[bands["nse_symbol"].eq(SYMBOL)])
review = audit.loc[audit["large_daily_move"] | audit["note"].str.strip().ne("")].copy()
print(f"{int(audit['large_daily_move'].sum())} events have large daily moves; {len(review)} events have a flag or source note")
scroll_table(review[["nse_symbol", "fiscal_quarter", "result_date", "weekly_move_pct",
                     "largest_daily_move_pct", "large_daily_move", "included", "note"]])

,nse_symbol,band_pct,events,close_inside_pct,range_inside_pct
259,RELIANCE,1.00,27,14.81,0.00
260,RELIANCE,2.00,27,22.22,0.00
261,RELIANCE,3.00,27,40.74,11.11
262,RELIANCE,4.00,27,62.96,14.81
263,RELIANCE,5.00,27,81.48,33.33
264,RELIANCE,7.50,27,92.59,77.78
265,RELIANCE,10.00,27,100.00,81.48


0 events have large daily moves; 76 events have a flag or source note


nse_symbol,fiscal_quarter,result_date,weekly_move_pct,largest_daily_move_pct,large_daily_move,included,note
ADANIPORTS,Q3 FY23,2023-02-07,4.35,8.33,False,True,Missing from NSE meeting records; taken from results filing date
APOLLOHOSP,Q3 FY23,2023-02-14,4.20,5.13,False,True,Missing from NSE meeting records; inferred as day before 15-Feb-2023 filing - verify
DRREDDY,Q4 FY20,2020-05-20,1.11,4.72,False,True,NSE meeting record showed 13-May-2020; results were filed 20-May-2020
ETERNAL,Q1 FY22,2021-08-10,11.26,8.35,False,True,Reported as Zomato
ETERNAL,Q2 FY22,2021-11-10,10.07,8.59,False,True,Reported as Zomato
ETERNAL,Q3 FY22,2022-02-10,-13.66,7.05,False,True,Reported as Zomato
ETERNAL,Q4 FY22,2022-05-23,30.94,14.50,False,True,Reported as Zomato
ETERNAL,Q1 FY23,2022-08-01,25.05,19.98,False,True,Reported as Zomato
ETERNAL,Q2 FY23,2022-11-10,0.55,13.84,False,True,Reported as Zomato
ETERNAL,Q3 FY23,2023-02-09,0.37,5.41,False,True,Reported as Zomato


## Export reusable results

Rerunning this cell replaces generated CSVs under `data/processed/option_selling/stocks/quarterly_results_weekly_move/7_trading_sessions/` by default (other horizons use their own subfolder). The audit contains every filtered input event; use `included_events.csv` for the analyzed sample. Prices and returns are exported at full precision. A JSON file records settings and selected source filenames. All dates are Asia/Kolkata session dates.

In [7]:
if EXPORT_CSV:
    output_name = f"{TRADING_SESSIONS}_trading_sessions" if WINDOW_MODE == "trading_sessions" else WINDOW_MODE
    output_dir = EXPORT_DIR / output_name
    output_dir.mkdir(parents=True, exist_ok=True)
    exports = {
        "source_coverage.csv": coverage,
        "all_events_audit.csv": audit,
        "included_events.csv": weekly,
        "stock_summary.csv": summary.reset_index(),
        "weekly_price_bands.csv": bands,
        "events_for_review.csv": review,
    }
    for filename, frame in exports.items():
        frame.to_csv(output_dir / filename, index=False)
    settings = {
        "window_mode": WINDOW_MODE, "trading_sessions": TRADING_SESSIONS, "start_result_date": START_RESULT_DATE,
        "end_result_date": END_RESULT_DATE, "large_daily_move_pct": LARGE_DAILY_MOVE_PCT,
        "exclude_large_daily_moves": EXCLUDE_LARGE_DAILY_MOVES, "bands_pct": BANDS_PCT,
        "result_dates_file": str(RESULTS_CSV.relative_to(PROJECT_ROOT)),
        "baseline": "last observed market close on or before result date",
        "market_calendar": "union of session dates in loaded stock files",
        "source_files": {symbol: source_files[symbol].name for symbol in prices},
    }
    (output_dir / "analysis_settings.json").write_text(json.dumps(settings, indent=2) + "\n")
    print(f"Saved {len(exports)} CSV files and analysis_settings.json to {output_dir.relative_to(PROJECT_ROOT)}")

Saved 6 CSV files and analysis_settings.json to data/processed/option_selling/stocks/quarterly_results_weekly_move/7_trading_sessions
